<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания № 22


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

**Описание задачи:**

Создать базовый класс Project в C#, который будет представлять проекты в рамках организации. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для демонстрации полиморфизма.

**Требования к базовому классу Project:**

* **Атрибуты:** ID проекта (ProjectId), Название проекта (ProjectName), Статус проекта (Status).
* **Методы:**
  * `UpdateStatus()`: метод для обновления статуса проекта.
  * `GetProjectDetails()`: метод для получения деталей проекта.
  * `AssignTeamMember()`: метод для назначения командного члена проекту.

**Требования к производным классам:**

1. **ПродуктовыйПроект (ProductProject):** Должен содержать дополнительные атрибуты, такие как Ожидаемая дата завершения (ExpectedCompletionDate). Метод `UpdateStatus()` должен быть переопределен для включения ожидаемой даты завершения в статус проекта.
2. **ИсследовательскийПроект (ResearchProject):** Должен содержать дополнительные атрибуты, такие как Финансирование (FundingAmount). Метод `AssignTeamMember()` должен быть переопределен для указания специфических требований к навыкам командного члена для исследовательского проекта.
3. **ИнфраструктурныйПроект (InfrastructureProject)** (если требуется третий класс): Должен содержать дополнительные атрибуты, такие как Срок выполнения (ExecutionPeriod). Метод `GetProjectDetails()` должен быть переопределен для отображения срока выполнения проекта вместе с другими деталями проекта.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
public delegate void ProjectStatusChangedHandler(Project sender, string oldStatus, string newStatus);
public delegate void ExpenseLoggedHandler(Project sender, decimal amount, string category);

public class Project
{
    private int _projectId;
    private string _projectName;
    private string _status;
    private int _progressPercentage;
    private decimal _budget;

    public event ProjectStatusChangedHandler OnStatusChanged;
    public event ExpenseLoggedHandler OnExpenseLogged;
    public event Action<Project> OnProjectCompleted;

    public int ProjectId
    {
        get => _projectId;
        protected set
        {
            if (value <= 0)
                throw new ArgumentException("ID проекта должен быть положительным числом!");
            _projectId = value;
        }
    }

    public string ProjectName
    {
        get => _projectName;
        protected set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Название проекта не может быть пустым!");
            _projectName = value;
        }
    }

    public string Status
    {
        get => _status;
        protected set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Статус не может быть пустым!");
            _status = value;
        }
    }

    public int ProgressPercentage
    {
        get => _progressPercentage;
        set
        {
            _progressPercentage = Math.Clamp(value, 0, 100);
            if (_progressPercentage == 100)
            {
                CompleteProject();
            }
        }
    }

    public decimal Budget
    {
        get => _budget;
        protected set
        {
            if (value < 0)
                throw new ArgumentException("Бюджет не может быть отрицательным!");
            _budget = value;
        }
    }

    public string Priority { get; set; }

    public HashSet<string> TechStack { get; protected set; }

    public List<string> TeamMembers { get; protected set; }

    public Project(int projectId, string projectName, decimal budget, string priority = "Средний", string status = "Создан")
    {
        ProjectId = projectId;
        ProjectName = projectName;
        Budget = budget;
        Priority = priority;
        Status = status;
        _progressPercentage = 0;
        TechStack = new HashSet<string>(StringComparer.OrdinalIgnoreCase);
        TeamMembers = new List<string>();
    }

    public virtual void UpdateStatus(string newStatus)
    {
        string oldStatus = Status;
        Status = newStatus;
        Console.WriteLine($"[Проект #{ProjectId} '{ProjectName}'] Смена статуса: {oldStatus} -> {newStatus}");

        OnStatusChanged?.Invoke(this, oldStatus, newStatus);
    }

    public virtual string GetProjectDetails()
    {
        string techs = TechStack.Count > 0 ? string.Join(", ", TechStack) : "стек не указан";
        return $"ID: {ProjectId} | '{ProjectName}' | Прогресс: {ProgressPercentage}% | Бюджет: {Budget:N0} руб. | Стек: [{techs}] | Статус: {Status}";
    }

    public virtual void AssignTeamMember(string member)
    {
        if (!string.IsNullOrWhiteSpace(member))
        {
            TeamMembers.Add(member);
            Console.WriteLine($"[Проект #{ProjectId}] Назначен участник: {member}");
        }
    }

    public bool AddTechnology(string technology)
    {
        bool added = TechStack.Add(technology);
        if (added)
        {
            Console.WriteLine($"[Проект '{ProjectName}'] В технологический стек добавлен: {technology}");
        }
        return added;
    }

    public void LogExpense(decimal amount, string category)
    {
        Console.WriteLine($"[Финансы '{ProjectName}'] Расход: {amount:N0} руб. (Категория: {category})");
        OnExpenseLogged?.Invoke(this, amount, category);
    }

    public void CompleteProject()
    {
        UpdateStatus("Завершен успешно");
        OnProjectCompleted?.Invoke(this);
    }

    public override bool Equals(object obj)
    {
        if (obj is Project other)
        {
            return ProjectId == other.ProjectId;
        }
        return false;
    }

    public override int GetHashCode()
    {
        return ProjectId.GetHashCode();
    }
}

public class ProductProject : Project
{
    public DateTime ExpectedCompletionDate { get; set; }
    public string ReleaseTarget { get; set; }

    public ProductProject(int projectId, string projectName, decimal budget, DateTime completionDate, string releaseTarget)
        : base(projectId, projectName, budget, "Высокий", "Разработка продукта")
    {
        ExpectedCompletionDate = completionDate;
        ReleaseTarget = releaseTarget;
    }

    public override void UpdateStatus(string newStatus)
    {
        base.UpdateStatus(newStatus);
        Console.WriteLine($"[Продукт '{ProjectName}'] Цель релиза: {ReleaseTarget}, Дедлайн: {ExpectedCompletionDate:dd.MM.yyyy}");
    }
}

public class ResearchProject : Project
{
    public decimal FundingAmount { get; set; }
    public string ScientificField { get; set; }

    public ResearchProject(int projectId, string projectName, decimal budget, decimal funding, string field)
        : base(projectId, projectName, budget, "Средний", "НИОКР")
    {
        FundingAmount = funding;
        ScientificField = field;
    }

    public override void AssignTeamMember(string member)
    {
        string specialized = $"{member} (Исследователь: {ScientificField})";
        TeamMembers.Add(specialized);
        Console.WriteLine($"[НИОКР '{ProjectName}'] Добавлен научный сотрудник: {specialized}");
    }
}

public class InfrastructureProject : Project
{
    public int ServerCount { get; set; }

    public InfrastructureProject(int projectId, string projectName, decimal budget, int serverCount)
        : base(projectId, projectName, budget, "Критический", "Инфраструктура")
    {
        ServerCount = serverCount;
    }
}

ProductProject webPortal = new ProductProject(1, "Клиентский Портал", 2500000, DateTime.Now.AddMonths(4), "Web & PWA");
ResearchProject aiAssistant = new ResearchProject(2, "AI Персональный Помощник", 4000000, 1500000, "GenAI");
InfrastructureProject k8sCluster = new InfrastructureProject(3, "Kubernetes Кластер", 1800000, 48);

webPortal.OnStatusChanged += (sender, oldStatus, newStatus) =>
    Console.WriteLine($"[СОБЫТИЕ: СМЕНА СТАТУСА] Проект '{sender.ProjectName}' переведен из '{oldStatus}' в '{newStatus}'!");

webPortal.OnExpenseLogged += (sender, amount, category) =>
    Console.WriteLine($"[СОБЫТИЕ: БУХГАЛТЕРИЯ] Проект '{sender.ProjectName}' списал {amount:N0} руб. на '{category}'.");

webPortal.OnProjectCompleted += (sender) =>
    Console.WriteLine($"[СОБЫТИЕ: УСПЕХ] УРА! Проект '{sender.ProjectName}' полностью завершен (100%)!");

Console.WriteLine("=== 1. ПРОВЕРКА СОБЫТИЙ И ДЕЛЕГАТОВ ===");
webPortal.AddTechnology("C#");
webPortal.AddTechnology(".NET 10");
webPortal.AddTechnology("PostgreSQL");
webPortal.AddTechnology("Docker");

bool isDuplicateAdded = webPortal.AddTechnology("C#");
Console.WriteLine($"Был ли добавлен дубликат технологии 'C#' в HashSet? {isDuplicateAdded}");

Console.WriteLine();
webPortal.LogExpense(120000, "Оплата хостинга");
webPortal.UpdateStatus("Тестирование MVP");
webPortal.ProgressPercentage = 100;

Console.WriteLine();
Console.WriteLine("=== 2. РАБОТА С КОЛЛЕКЦИЯМИ: List<Project> И LINQ С ЛЯМБДА-ВЫРАЖЕНИЯМИ ===");
List<Project> projectList = new List<Project> { webPortal, aiAssistant, k8sCluster };

aiAssistant.AddTechnology("Python");
aiAssistant.AddTechnology("PyTorch");
aiAssistant.AddTechnology("C#");
aiAssistant.ProgressPercentage = 65;

k8sCluster.AddTechnology("Kubernetes");
k8sCluster.AddTechnology("Linux");
k8sCluster.AddTechnology("Docker");
k8sCluster.ProgressPercentage = 80;

Console.WriteLine("Проекты с бюджетом >= 2 000 000 руб.:");
var highBudgetProjects = projectList.Where(p => p.Budget >= 2000000).OrderByDescending(p => p.Budget);
foreach (var p in highBudgetProjects)
{
    Console.WriteLine($"- {p.ProjectName}: {p.Budget:N0} руб. (Прогресс: {p.ProgressPercentage}%)");
}

Console.WriteLine();
Console.WriteLine("=== 3. РАБОТА С Dictionary<int, Project> (БЫСТРЫЙ ПОИСК ПО ID) ===");
Dictionary<int, Project> projectDictionary = new Dictionary<int, Project>
{
    { webPortal.ProjectId, webPortal },
    { aiAssistant.ProjectId, aiAssistant },
    { k8sCluster.ProjectId, k8sCluster }
};

int searchId = 2;
if (projectDictionary.TryGetValue(searchId, out Project found))
{
    Console.WriteLine($"Найден проект по ID {searchId}:");
    Console.WriteLine(found.GetProjectDetails());
}

Console.WriteLine();
Console.WriteLine("=== 4. РАБОТА С HashSet<Project> (УНИКАЛЬНОСТЬ ПРОЕКТОВ) ===");
HashSet<Project> uniqueProjects = new HashSet<Project>();
uniqueProjects.Add(webPortal);
uniqueProjects.Add(aiAssistant);
uniqueProjects.Add(k8sCluster);

ProductProject duplicateProject = new ProductProject(1, "Клон Портала", 999999, DateTime.Now, "Clone");
bool addedDuplicate = uniqueProjects.Add(duplicateProject);
Console.WriteLine($"Удалось ли добавить проект с уже существующим ID 1 в HashSet? {addedDuplicate}");
Console.WriteLine($"Всего уникальных проектов в множестве: {uniqueProjects.Count}");
